In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/25 04:46:34 WARN Utils: Your hostname, whizdom-VirtualBox, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/02/25 04:46:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 04:46:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-02-25 04:47:09--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-25T04%3A42%3A17Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-02-25T03%3A41%3A54Z&ske=2026-02-25T04%3A42%3A17Z&sks=b&skv=2018-11-09&sig=HHb4wSdgcNqt9OKe5fI9uY3V6WeBuzDSqy5a67HX4mc%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MTk5NDgzMSwibmJmIjoxNzcxOTkxMjMxLCJwYXRoIjoi

In [4]:
!gzip -dc fhvhv_tripdata_2021-01.csv.gz | wc -l

11908469


## Note
Doing a bit of research, I found out that Spark can read compressed CSV files directly. You definitely do not need to uncompress them first.

When I run spark.read.csv('file.csv.gz'), Spark is smart enough to look at the file extension. If it sees .gz, it automatically uses the correct codec to decompress the data in memory as it reads it.

However, Gzip is not splittable, hence one file will be processed by a single spark core (task) even if there is a massive cluster. That is, Spark would not be able to parallelize the reading process. The solution to this then is to convert the CSV into a parquet format. Parquet files are compressed and splittable, allowing spark to distribute the work across all available CPU cores

In [5]:
df = spark.read \
    .option("header", "true") \
    .csv("fhvhv_tripdata_2021-01.csv.gz")

In [6]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

## NOTES

In Spark, a DataFrame schema is essentially a `StructType` thta contains a collection of `StructField` objects. Each field has a name (which in this case is the column name), a datatype, and a flag for whether it can be NULL or not.
Spark types are majorly divided into Atomic (Simple) and Complex (nesteed) types. 

Below is the Spark Data Type Heirachy. Each type can be gotten from the `types` module in the `pyspark.sql` package 

| Type Category | Spark Type | Description |
| :--- | :--- | :--- |
| Numeric | `IntegerType` | 4-byte signed integers (most IDs, counts). |
| Numeric | `LongType` | 8-byte signed integers (large IDs, epoch timestamps). |
| Numeric | `FloatType` | 4-byte single-precision floating point. |
| Numeric | `DoubleType` | 8-byte double-precision (Standard for Fare/Amount). |
| String | `StringType` | Character strings (Vendor names, Store IDs). |
| Binary | `BinaryType` | Binary data (Images, serialized objects). |
| Boolean | `BooleanType` | True / False values. |
| Datetime | `TimestampType` | Date + Time (Standard for Pick-up/Drop-off). |
| Datetime | `DateType` | Date only (YYYY-MM-DD). |
| Complex | `ArrayType` | A list of values (e.g., Array[Integer]). |
| Complex | `MapType` | Key-value pairs. |
| Complex | `StructType` | A nested object (a DataFrame within a column). |

In [7]:
from pyspark.sql import types

```
StructType([
    StructField('hvfhs_license_num', StringType(), True), 
    StructField('dispatching_base_num', StringType(), True), 
    StructField('pickup_datetime', StringType(), True), 
    StructField('dropoff_datetime', StringType(), True), 
    StructField('PULocationID', StringType(), True), 
    StructField('DOLocationID', StringType(), True), 
    StructField('SR_Flag', StringType(), True)
])
```

## Let us define the schema

In [9]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

## Reading in the data again but with the correct schema

In [10]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("fhvhv_tripdata_2021-01.csv.gz")

In [11]:
df = df.repartition(24)

In [12]:
df.write.parquet('fhvhv/2021/01/')

In [18]:
!du -h fhvhv/2021/01/

228M	fhvhv/2021/01/


In [19]:
!tree fhvhv/2021/01/

fhvhv/2021/01/
├── part-00000-a2cf3bc0-680f-4bd4-8482-fdc29984bd79-c000.snappy.parquet
├── part-00001-a2cf3bc0-680f-4bd4-8482-fdc29984bd79-c000.snappy.parquet
├── part-00002-a2cf3bc0-680f-4bd4-8482-fdc29984bd79-c000.snappy.parquet
├── part-00003-a2cf3bc0-680f-4bd4-8482-fdc29984bd79-c000.snappy.parquet
├── part-00004-a2cf3bc0-680f-4bd4-8482-fdc29984bd79-c000.snappy.parquet
├── part-00005-a2cf3bc0-680f-4bd4-8482-fdc29984bd79-c000.snappy.parquet
├── part-00006-a2cf3bc0-680f-4bd4-8482-fdc29984bd79-c000.snappy.parquet
├── part-00007-a2cf3bc0-680f-4bd4-8482-fdc29984bd79-c000.snappy.parquet
├── part-00008-a2cf3bc0-680f-4bd4-8482-fdc29984bd79-c000.snappy.parquet
├── part-00009-a2cf3bc0-680f-4bd4-8482-fdc29984bd79-c000.snappy.parquet
├── part-00010-a2cf3bc0-680f-4bd4-8482-fdc29984bd79-c000.snappy.parquet
├── part-00011-a2cf3bc0-680f-4bd4-8482-fdc29984bd79-c000.snappy.parquet
├── part-00012-a2cf3bc0-680f-4bd4-8482-fdc29984bd79-c000.snappy.parquet
├── part-00013-a2cf3bc0-680f-4bd4-8482-fdc29984bd

In [24]:
# Now let us read in these parquet file
df = spark.read.parquet('fhvhv/2021/01/')

In [25]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [32]:
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)
spark.conf.set('spark.sql.repl.eagerEval.maxNumRows', 5)

In [33]:
df

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02617,2021-01-21 18:19:13,2021-01-21 19:14:27,256,265,NULL
HV0003,B02871,2021-01-26 11:15:51,2021-01-26 11:23:14,165,165,NULL
HV0003,B02764,2021-01-19 19:11:47,2021-01-19 19:29:45,82,137,NULL
HV0003,B02872,2021-01-27 11:49:13,2021-01-27 11:53:15,97,97,NULL
HV0005,B02510,2021-01-27 15:21:49,2021-01-27 15:29:59,134,95,NULL


In [34]:
# Pyspark Functions
from pyspark.sql import functions as F

In [35]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [36]:
crazy_stuff('B02884')

's/b44'

In [37]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [38]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

[Stage 11:>                                                         (0 + 1) / 1]

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/a39| 2021-01-21|  2021-01-21|         256|         265|
|  a/b37| 2021-01-26|  2021-01-26|         165|         165|
|  e/acc| 2021-01-19|  2021-01-19|          82|         137|
|  e/b38| 2021-01-27|  2021-01-27|          97|          97|
|  e/9ce| 2021-01-27|  2021-01-27|         134|          95|
|  s/acd| 2021-01-17|  2021-01-17|          74|         116|
|  e/b38| 2021-01-28|  2021-01-28|          78|          75|
|  a/a7a| 2021-01-04|  2021-01-04|         236|         126|
|  e/9ce| 2021-01-21|  2021-01-21|          79|         137|
|  e/b3c| 2021-01-29|  2021-01-29|         198|         157|
|  e/acc| 2021-01-24|  2021-01-24|          37|          37|
|  e/acc| 2021-01-05|  2021-01-05|         106|          72|
|  e/acc| 2021-01-20|  2021-01-20|          35|         265|
|  e/a39| 2021-01-10|  2

In [39]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003')



pickup_datetime,dropoff_datetime,PULocationID,DOLocationID
2021-01-21 18:19:13,2021-01-21 19:14:27,256,265
2021-01-26 11:15:51,2021-01-26 11:23:14,165,165
2021-01-19 19:11:47,2021-01-19 19:29:45,82,137
2021-01-27 11:49:13,2021-01-27 11:53:15,97,97
2021-01-17 18:35:49,2021-01-17 18:55:43,74,116
